In [13]:
from langchain_openai import ChatOpenAI
from langchain_community.chat_models import ChatOllama
from pydantic import Field, BaseModel
from langchain_core.prompts import ChatPromptTemplate
from typing import List
import json
from dotenv import load_dotenv

load_dotenv()

class Product(BaseModel):
    """Structured product review analysis"""
    product_name:str = Field(description="Name of the product under review")
    sentiments:str = Field(description="Overall sentiment: positive, negative, neutral")
    ratings: int = Field(description="Rating from 1 -5", ge=1, le=5)
    pros:List[str] = Field(description="List of positive aspect")
    cons:List[str] = Field(description="List of negative aspect")
    summary:str = Field(description="Brief summary of review")

llm = ChatOpenAI(model="gpt-4o-mini")
# llm = ChatOllama(model="mixtral")

structured_llm = llm.with_structured_output(Product)

prompting = ChatPromptTemplate([
    ("system","You are a product review analyzer. Extract structured  information from review"),
    ("user","{review_request}")
])

review_prompt = """
I bought a Mono Tech Phone with 64GB memory and a 120hz screen resolution. This phone works well, and has a very bright screen
that is visible even in the sun. The weight compared to the iphone is lighter and easier to carry. It runs a the latest android OS
and browsing on it is seamless. The only issue I have with this phone is that it does not have a long lasting battery life like the
iphone. Other than that, I think its perfect for its price which is like half that of iphone
"""
chain = prompting | structured_llm

response = chain.invoke({
    "review_request": review_prompt
})


c:\lang1\venvi\Lib\site-packages\pydantic\main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=Product(product_name='Mon... in battery longevity.'), input_type=Product])
  return self.__pydantic_serializer__.to_python(


In [14]:
print(json.dumps(response.model_dump(), indent= 2))

{
  "product_name": "Mono Tech Phone",
  "sentiments": "positive",
  "ratings": 4,
  "pros": [
    "Bright screen visible in sunlight",
    "Lightweight compared to iPhone",
    "Latest Android OS",
    "Seamless browsing experience",
    "Great value for its price"
  ],
  "cons": [
    "Short battery life compared to iPhone"
  ],
  "summary": "The Mono Tech Phone offers great features for its price, including a bright display and lightweight design, though it lacks in battery longevity."
}
